
<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Imagenes/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Propedéutico en Programación y Algoritmos para la Maestría en Ciencia de Datos</h1>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Módulo 3: Manipulación de Datos con la librería Pandas II</h3>
    </div>
</div>


<img style="float: right; margin: 0px 0px 15px 15px;" src="https://numfocus.org/wp-content/uploads/2016/07/pandas-logo-300.png" width="400px" height="400px" />

> La clase pasada comenzamos a ver las funcionalidades de la librería [pandas](https://pandas.pydata.org/). Básicamente, aprendimos a cargar datos desde archivos `.csv`, aprendimos a seleccionar subcojuntos de estos datos mediante indización (obtener ciertas filas, o ciertas columnas, o ciertas filas y columnas determinadas). Finalmente, aprendimos a filtrar datos, es decir, a seleccionar registros de la base de datos que satisfagan cierta condición.

> Hoy nos enfocaremos a ver como reunir varias bases de datos que poseen información complementaria relativa a un mismo problema, para obtener una sola tabla con la información condensada.

Referencias:
- https://www.shanelynn.ie/merge-join-dataframes-python-pandas-index-1/
___

# 0. Motivación

En cualquier situación relacionada con ciencia de datos en la vida real, no pasarán más de 10 minutos sin que aparezca la necesidad de unir (`merge` o `join`) dos bases de datos complementarias para formar tu conjunto de datos para el análisis.

La unión de DataFrames es un proceso fundamental, que cualquier analista de datos en formación debe aprender. En esta clase aprenderemos cómo hacerlo, y en el camino responderemos las siguientes preguntas:

- ¿Qué son el `merge` o `join` de dos DataFrames?

- ¿Qué son `inner`, `outer`, `left` y `right` `joins`?



## Datos para la clase

En esta clase, seguiremos trabajando con los datos de la clase pasada:

En la carpeta "data" tenemos los archivos 

- "customers_data.csv": datos propios de los clientes de la empresa, 
- "sessions_data.csv": inicios de sesión de los clientes en la plataforma web de compras (similar a amazon),
- "transactions_data.csv": transacciones asociadas a cada inicio de sesión, y
- "products_data.csv": información de los productos. 

Podemos cargar cada uno de los archivos `.csv` como DataFrames de Pandas usando la función `read_csv()` de pandas, y examinar los contenidos de cada uno con el método `head()`.

In [ ]:
# Importar pandas
import pandas as pd

In [ ]:
# Cargar customers_data
customers = pd.read_csv('Data/customers_data.csv', index_col=[0])
customers

In [ ]:
customers.shape

In [ ]:
# Cargar sessions_data
sesions = pd.read_csv('Data/sessions_data.csv', index_col=[0])
sesions

In [ ]:
# Cargar transactions_data
transactions = pd.read_csv('Data/transactions_data.csv', index_col=[0])
transactions

In [ ]:
# Cargar products_data
products = pd.read_csv('Data/products_data.csv', index_col=[0])
products

## Bonus: formato de fechas y horas

Como vimos la vez pasada, es relevante en general un correcto manejo de las fechas y horas en las bases de datos.

Lo primero que se tiene que hacer para un correcto manejo de las fechas (y horas) es identificar las columnas o variables que contienen fechas. Por ejemplo, de la tabla `customers_data`:

Lo que sigue, es especificarle a pandas el formato en que vienen esas fechas con la función `to_datetime()` de pandas

In [ ]:
# Cargar customers_data
customers = pd.read_csv('Data/customers_data.csv', index_col=[0])
customers

In [ ]:
# Ayuda en la función to_datetime


In [ ]:
# Especificar el formato de fechas en la tabla customers_data


In [ ]:
customers['join_date'] = pd.to_datetime(customers['join_date'], format='%Y-%m-%d %H:%M:%S', errors = 'coerce')
customers['join_date']

In [ ]:
customers

In [ ]:
customers['date_of_birth']=pd.to_datetime(customers['date_of_birth'], format='%Y-%m-%d', errors = 'coerce')
customers['date_of_birth']

Hagan esto mismo para todas las tablas:

In [ ]:
sesions.head()

In [ ]:
# Especificar el formato de fechas en la tabla sessions


In [ ]:
# Especificar el formato de fechas en la tabla customers
transactions['transaction_time'] = pd.to_datetime(transactions['transaction_time'], format='%Y-%m-%d %H:%M:%S', errors = 'coerce')
transactions['transaction_time']

## Volviendo al problema ...

Hay columnas o variables que relacionan las tablas. Por ejemplo:

- Los clientes pueden iniciar sesión en la plataforma cuantas veces quieran. De manera que hay una relación uno a muchos entre las tablas "customers_data" y "sessions_data", mediante la variable "customer_id".

- Cuando se efectúa una transacción se supone que se está comprando un producto. Por lo tanto hay una relación uno a muchos entre las tablas "transactions_data"  y "products_data", mediante la variable "product_id".

## Problemas

Primero, quisieramos determinar cuál es el dispositivo preferido por zona.

Luego, quisiéramos determinar cuál es la marca preferida por zona.

Para resolver lo anterior necesitamos unir ("merge" o "join") nuestros datasets en uno solo para el análisis.

___
# 1. Uniendo dos DataFrames...

> “Merging” two datasets is the process of bringing two datasets together into one, and aligning the rows from each based on common attributes or columns.

Las palabras “merge” y “join” se usan indistintamente en Pandas, y en otros lenguajes como SQL y R. En Pandas, hay métodos separados “merge” y “join”, que realizan cosas similares (personalmente uso el método "merge"), y la función "merge".

Vamos a concentrarnos en el **primer problema**. En este escenario, necesitamos llevar a cabo los siguientes pasos:

- Para cada fila en el dataset `sessions_data`, debemos hacer una nueva columna que contenga el "zip_code" respectivo de cada cliente.
- Una vez hagamos esto, obtenemos la moda de los dispositivos para cada cliente.

**¿Podemos usar un ciclo?**

Claro que si. Se podría escribir un ciclo para esta tarea. Éste iría a través de cada fila en `sessions_data`, y a cada "user_id" asignar el valor de la nueva columna con la zona respectiva.

Sin embargo, usar ciclos haría nuestra tarea mucho más lenta y plagada de más código que el necesario que si se usara la función (método) `join()`.

De forma que, para estas situaciones, **nunca usar ciclos**.

Para ver cómo podemos hacer lo anterior, veamos la ayuda de la función `merge()` de pandas:

In [ ]:
# Ayuda de merge()
help(pd.merge)

Ahora, veamos como podemos añadir el código postal a la tabla `sessions_data`, usando la función `merge()` de pandas:

In [ ]:
# Uso de la función merge()
# sessions_with_zip = pd.merge(left=sessions,
#                              right=customers[['customer_id', 'zip_code']]
#                              on="customer_id")
sesions_with_zip = pd.merge(left=sesions, right=customers[['customer_id', 'zip_code']], on = 'customer_id', how = 'inner')
sesions_with_zip

La función `merge()` es el objetivo principal de esta clase. Básicamente, la operación de unir dos DataFrames hace lo siguiente: 

- toma el DataFrame de la izquierda (argumento left=), 
- el DataFrame de la derecha (argumento right=),
- la columna donde se va a unir (argumento on=), y
- la forma en que se va a unir (argumento how=).

La variable en común sobre la que se hace la unión está especificada en el argumento `on`.

Con este resultado, podemos filtrar por zona y luego obtener la moda.

In [ ]:
# Obtener la moda por zona


### Usando GroupBy

### ¿Qué son los tipos de unión inner, left, right y outer?

En nuestro ejemplo, unimos `sessions` con `customers` sobre la columna "custumers_id". En este caso, en ambas tablas existían todos los posibles valores de "customers_id" (1, 2, 3, 4, 5).

¿Qué pasa si esta situación no se cumple?

Por ejemplo, realicemos los siguientes cambios artificiales:

Por defecto, la operación `merge()` de pandas actpua con un merge tipo "inner". Un "inner merge", guarda únicamente los valores comúnes (en la columna especificada en el argumento `on=`) de ambos DataFrames.

Por ejemplo:

Los otros comportamientos se pueden revisar mejor a través de ejemplos:

In [ ]:
# left join
sesions_with_zip_art = pd.merge(left=sesions_art, right=customers_art, on = 'customer_id', how = 'left')
sesions_with_zip_art.head()


In [ ]:
sesions_with_zip_art['customer_id'].unique()

In [ ]:
# right join
sesions_with_zip_art = pd.merge(left=sesions_art, right=customers_art, on = 'customer_id', how = 'right')
sesions_with_zip_art.head()


In [ ]:
sesions_with_zip_art['customer_id'].unique()

In [ ]:
# outer join
sesions_with_zip_art = pd.merge(left=sesions_art, right=customers_art, on = 'customer_id', how = 'outer')
sesions_with_zip_art.head()


In [ ]:
sesions_with_zip_art['customer_id'].unique()

In [ ]:
sesions_with_zip_art

Estos comportamientos son muy intuitivos a partir de sus valores.

Así mismo, sus usos son muy intuitivos (lo sabrán cuando se enfrenten a problemas reales).

### Resolvamos el problema 2

Determinar cuál es la marca preferida por zona.